# SQL and Python: pandas, SQLite and Jupyter magic

Create a small fictional energy dataset, save it to SQLite, query it with Python and pandas, and then use SQL directly in notebook cells. Run the cells in order in a local Jupyter kernel. The first part needs pandas and Python's built-in `sqlite3`; the optional magic section also needs `ipython-sql`. Values are illustrative, not observations.

The tutorial creates `energy_tutorial.db` in the notebook's **current working directory** and replaces only its `generation_tutorial` table on rerun. Use a separate tutorial folder, not a production database. `if_exists="replace"` deletes that table and its contents. The generated table is for learning; define keys and constraints explicitly for production data.

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path

df = pd.DataFrame({
    "country": ["Austria", "Austria", "Denmark", "Denmark", "Austria"],
    "source": ["Hydro", "Solar", "Wind", "Solar", "Wind"],
    "year": [2023, 2023, 2023, 2024, 2024],
    "generation_gwh": [45.0, 12.5, 20.0, 15.0, None],
})
df

Keep `pd` as the pandas module alias; use `df` for the DataFrame. `None` becomes a missing value. Units are GWh per row, and the examples are deliberately small.

In [ ]:
database_path = Path("energy_tutorial.db")
connection = sqlite3.connect(database_path)
try:
    df.to_sql("generation_tutorial", connection, if_exists="replace", index=False)
    connection.commit()
    print("Tutorial database:", database_path.resolve())
    print("Rows written:", connection.execute("SELECT COUNT(*) FROM generation_tutorial").fetchone()[0])
finally:
    connection.close()

`sqlite3.connect` opens or creates a local file; `to_sql` creates and loads a table. Closing the connection releases resources. For larger or shared applications, choose an appropriate server DBMS and configure access separately.

In [ ]:
# Reopen the same database and return a query as a DataFrame.
connection = sqlite3.connect(database_path)
try:
    result = pd.read_sql_query(
        "SELECT country, source, year, generation_gwh FROM generation_tutorial WHERE year = ? ORDER BY country, source",
        connection,
        params=(2023,),
    )
finally:
    connection.close()
result

The `?` placeholder binds the value safely instead of constructing SQL with string interpolation. SQL identifiers such as table names are different: use fixed trusted names, not unvalidated user input. The connection is closed after reading.

In [ ]:
connection = sqlite3.connect(database_path)
try:
    summary = pd.read_sql_query("""
        SELECT source, COUNT(*) AS rows, COUNT(generation_gwh) AS reported,
               SUM(generation_gwh) AS total_gwh, AVG(generation_gwh) AS average_gwh
        FROM generation_tutorial GROUP BY source ORDER BY source
    """, connection)
finally:
    connection.close()
summary

`COUNT(*)` counts rows and `COUNT(generation_gwh)` counts only non-null measurements; `SUM` and `AVG` ignore nulls. The resulting sums are illustrative across different countries and years, not an official energy total. Always report coverage and compare compatible units and periods.

## Jupyter magic commands

A line magic begins with `%` and acts on the remainder of one line. A cell magic begins with `%%` on the **first line** and interprets the whole cell. These commands belong to IPython/Jupyter, not ordinary Python scripts or SQL itself. `%pip` installs into the active kernel; `!pip` runs a shell command and may target a different Python environment. See [IPython magics](https://ipython.readthedocs.io/en/stable/interactive/magics.html).

For a local Jupyter kernel, run this once in a separate cell if needed, then restart the kernel and rerun the cells above:

```ipython
%pip install ipython-sql "prettytable<3.12"
```

`!pip install ipython-sql "prettytable<3.12"` is the shell alternative, but `%pip` is preferred. The PrettyTable constraint avoids a `KeyError: DEFAULT` with newer PrettyTable releases and ipython-sql 0.5.0. The `ipython-sql` project describes itself as legacy and points to JupySQL; the following syntax is included to teach the requested extension. Browser-based JupyterLite/Pyodide may not support installing this package or accessing the same filesystem: use the Python/pandas examples above or a local Jupyter environment. [ipython-sql documentation](https://github.com/catherinedevlin/ipython-sql).

In [ ]:
%load_ext sql

Connect to the **same file in the same working directory**. Three slashes mean a relative SQLite file; `sqlite:///energy_tutorial.db` is a SQLAlchemy URL, not `salute://`. An absolute Unix path has four slashes. Do not paste credentials into a public notebook.

In [ ]:
%sql sqlite:///energy_tutorial.db

In [ ]:
%%sql
SELECT COUNT(*) AS all_rows, COUNT(generation_gwh) AS reported_rows
FROM generation_tutorial;

In [ ]:
%%sql
SELECT country, source, year, generation_gwh,
       generation_gwh * 1000 AS generation_mwh
FROM generation_tutorial
WHERE year >= 2023
ORDER BY year, country, source;

In [ ]:
%%sql
SELECT source, COUNT(*) AS rows, COUNT(generation_gwh) AS reported,
       AVG(generation_gwh) AS mean_gwh
FROM generation_tutorial
GROUP BY source
ORDER BY source;

Try `WHERE generation_gwh IS NULL`, `MIN`, `MAX`, and a different year. Start with [SQL basics](sql-basics.md), then use the [renewable-energy SQL reference](SQL.md) and the [Eurostat notebook](SQL_Pandas.ipynb). Close the notebook's SQL connection or restart the kernel before deleting the tutorial file.